In [42]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Understand Caching")
    .master("local[*]")
    .config("spark.executor.memory", "512M")
    .getOrCreate()
)

spark

In [44]:
# Read Sales CSV Data - 752MB Size ~ 7.2M Records

_schema = "transacted_at string, trx_id string, retailer_id string, description string, amount double, city_id string"

df = spark.read.format("csv").schema(_schema).option("header", True).load("data/input/new_sales.csv")

In [5]:
df.where("amount > 300").show()

+--------------------+----------+-----------+--------------------+-------+----------+
|       transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+--------------------+----------+-----------+--------------------+-------+----------+
|2017-11-24T19:00:...|1734117022|  847200066|Wal-Mart  ppd id:...|1737.26|1646415505|
|2017-11-24T19:00:...|1734117030| 1953761884|Home Depot     pp...|  384.5| 287177635|
|2017-11-24T19:00:...|1734117153|  847200066|unkn        Kings...|2907.57|1483931123|
|2017-11-24T19:00:...|1734117241|  486576507|              iTunes|2912.67|1663872965|
|2017-11-24T19:00:...|2076947146|  511877722|unkn     ccd id: ...|1915.35|1698762556|
|2017-11-24T19:00:...|2076947113| 1996661856|AutoZone  arc id:...| 1523.6|1759612211|
|2017-11-24T19:00:...|2076946994| 1898522855|Target    ppd id:...|2589.93|2074005445|
|2017-11-24T19:00:...|2076946121|  562903918|unkn    ccd id: 5...| 315.86|1773943669|
|2017-11-24T19:00:...|2076946063| 1070485878|Amazon.co

In [45]:
# Marks the DataFrame for caching (lazy)
df.cache()

# Must trigger an action/count/write
df.count()

# Default storage level for DataFrame and Dataset: MEMORY_AND_DISK
# For RDDs: MEMORY_ONLY

7202569

In [20]:
# Cache DataFrame (cache)
# Uses the default storage level

df_cache = df.where("amount > 100").cache()

In [21]:
df_cache.count()

2549058

In [22]:
df.where("amount > 50").show()

+--------------------+----------+-----------+--------------------+-------+----------+
|       transacted_at|    trx_id|retailer_id|         description| amount|   city_id|
+--------------------+----------+-----------+--------------------+-------+----------+
|2017-11-24T19:00:...|1995601912| 2077350195|Walgreen       11-25| 197.23| 216510442|
|2017-11-24T19:00:...|1734117022|  847200066|Wal-Mart  ppd id:...|1737.26|1646415505|
|2017-11-24T19:00:...|1734117030| 1953761884|Home Depot     pp...|  384.5| 287177635|
|2017-11-24T19:00:...|1734117089| 1898522855| Target        11-25|  66.33|1855530529|
|2017-11-24T19:00:...|1734117117|  997626433|Sears  ppd id: 85...| 298.87| 957346984|
|2017-11-24T19:00:...|1734117153|  847200066|unkn        Kings...|2907.57|1483931123|
|2017-11-24T19:00:...|1734117212| 1996661856|unkn    ppd id: 4...| 140.38| 336763936|
|2017-11-24T19:00:...|1734117241|  486576507|              iTunes|2912.67|1663872965|
|2017-11-24T19:00:...|2076947148|  847200066|Wal-Mart 

In [48]:
# Remove Cache

# Removes specific DataFrame/RDD from cache
df.unpersist()
df_cache.unpersist()

# Removes ALL cached tables and DataFrames
spark.catalog.clearCache()

In [49]:
# persist()
# More flexible; Lets you choose how to store data
# # MEMORY_ONLY, MEMORY_AND_DISK, MEMORY_ONLY_SER, MEMORY_AND_DISK_SER, DISK_ONLY, MEMORY_ONLY_2, MEMORY_AND_DISK_2

import pyspark

# df_persist = df.persist(pyspark.StorageLevel.MEMORY_ONLY)

df_persist_md = df.persist(pyspark.StorageLevel.MEMORY_AND_DISK)

In [50]:
df_persist_md.write.format("noop").mode("overwrite").save()

In [34]:
# PySpark Storage Levels (RDD + DataFrame)
#
# Each storage level controls:
# - Memory usage (RAM)
# - Disk usage
# - Serialization (only matters for RDDs)
# - Replication (suffix _2 = 2 copies)
#
# NOTE:
# - DataFrames always use optimized binary format internally (so _SER is mostly irrelevant)
# - RDDs: serialization makes a BIG difference (memory vs speed tradeoff)
#
# ------------------------------------------------------------
# MEMORY_ONLY
# - Store in RAM only
# - Fastest access
# - If not enough memory → recompute partitions
#
# MEMORY_AND_DISK
# - Store in RAM, spill overflow to disk
# - Safer (avoids recomputation)
# - Default for DataFrame.cache()
#
# MEMORY_ONLY_SER
# - Store serialized in RAM (less memory)
# - Slower (needs deserialization)
# - If not enough memory → recompute
#
# MEMORY_AND_DISK_SER
# - Serialized in RAM + spill to disk
# - Good for large datasets with limited memory
#
# DISK_ONLY
# - Store only on disk
# - No memory usage
# - Slowest but works for very large data
#
# MEMORY_ONLY_2
# - Same as MEMORY_ONLY but replicated on 2 nodes
# - Better fault tolerance, double memory usage
#
# MEMORY_AND_DISK_2
# - Same as MEMORY_AND_DISK but replicated
#
# DISK_ONLY_2
# - Disk storage with 2 replicas
#
# MEMORY_AND_DISK_SER_2
# - Serialized + disk spill + replicated
#
# OFF_HEAP (advanced)
# - Store outside JVM heap (requires config)
# - Reduces GC pressure
#
# ------------------------------------------------------------
# Quick selection guide:
# - Small data → MEMORY_ONLY
# - General use → MEMORY_AND_DISK
# - Memory issues → MEMORY_ONLY_SER
# - Large data → MEMORY_AND_DISK_SER or DISK_ONLY
# - Critical workloads → use *_2 (replication)
#
# cache() = persist() with default storage level
# Always trigger an action (count, show, etc.) to materialize cache

In [ ]:
# cache = deserialized
# persist = serialized

In [35]:
spark.stop()

In [40]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel

spark = SparkSession.builder \
    .appName("RDD StorageLevel Example") \
    .getOrCreate()

sc = spark.sparkContext

rdd = sc.parallelize(range(1, 11))  # 1 to 10

rdd_cache = rdd.cache()  # shorthand for MEMORY_ONLY
print("Cached RDD (MEMORY_ONLY) count:", rdd_cache.count())  # triggers caching

rdd_persist = rdd.persist(StorageLevel.MEMORY_AND_DISK)
print("Persisted RDD (MEMORY_AND_DISK) sum:", rdd_persist.sum())  # triggers caching

rdd_persist_ser = rdd.persist(StorageLevel.MEMORY_ONLY_SER)
print("Persisted RDD (MEMORY_ONLY_SER) max:", rdd_persist_ser.max())

rdd_cache.unpersist()        # remove from cache
rdd_persist.unpersist()      # remove from cache
rdd_persist_ser.unpersist()  # remove from cache

# RDD type: Python RDDs (PySpark) are always serialized because Python objects can’t be directly stored as JVM objects.
# Scala/Java RDDs: true MEMORY_ONLY = deserialized Java objects (fast).
# PySpark: MEMORY_ONLY behaves like MEMORY_ONLY_SER internally — data is serialized to JVM to cross Python-JVM boundary.

Persisted RDD (MEMORY_AND_DISK) sum: 55


PythonRDD[1] at RDD at PythonRDD.scala:53

In [41]:
spark.stop()